# BN â†’ EN Bulk Translation (Qwen2.5-3B-Instruct, base model)
Takes an input file â€” either a `test_sample_stratified*.parquet` (from the
sample-creation notebook) or an `.xlsx` with columns `ben`, `ref_en`,
`source` â€” normalizes it, runs every row through **Qwen2.5-3B-Instruct**
(zero-shot, no fine-tuning), and saves an output Excel file with columns:
`ben`, `ref_en`, `source`, `translated_en`.

**Before running:**
1. Kaggle â†’ Settings â†’ Accelerator â†’ **GPU T4 x2** (needed for the model).
2. Kaggle â†’ Add-ons â†’ Secrets: `HF_TOKEN`.
3. Internet: **On**.
4. Attach your input file via **Add Data â†’ Upload** (either the `.parquet`
   from the sample-creation notebook, or your own `.xlsx`), then set
   `INPUT_FILE` below to its path under `/kaggle/input/...`.


## 1. Install dependencies

In [ ]:
import glob as _g, json as _j, subprocess, sys

# Read model choice early so we can install the right packages
# before transformers is imported anywhere.
_rc = sorted(_g.glob("/kaggle/input/*/run_config.json"))
_mc = _j.load(open(_rc[-1])).get("MODEL_CHOICE", "Qwen2.5_3B") if _rc else "Qwen2.5_3B"
print(f"MODEL_CHOICE detected at install time: {_mc}")

packages = ["pandas", "pyarrow", "openpyxl", "accelerate", "bitsandbytes", "sentencepiece", "sacremoses"]
if _mc == "Indictrans2_1B":
    # IndicTrans2 remote code imports transformers.onnx, which was removed in newer transformers releases.
    packages.insert(0, "transformers==4.39.3")
else:
    packages.insert(0, "transformers")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=False)
print("Dependencies installed.")


## 2. Secrets + auth

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as _e:
    HF_TOKEN = None
    print(f"Warning: Could not load HF_TOKEN ({_e})")
    print("Proceeding without authentication — works for public models like Qwen2.5-3B-Instruct.")

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Authenticated with HuggingFace.")
else:
    print("No HF auth — using public model access.")


## 3. Set your input file path

Point this at the file you attached under **Add Data**. Accepts either:
- a `.parquet` file (e.g. `test_sample_stratified__from__....parquet`)
- an `.xlsx` file with columns `ben`, `ref_en` (optional), `source` (optional)


In [ ]:
# --- Automation override (added for UI orchestration) ---
import json
import os as _os
import glob as _glob

_RUN_CONFIG_PATH = "/kaggle/input/*/run_config.json"
_matches = sorted(_glob.glob(_RUN_CONFIG_PATH))

_auto_run_label = None
_auto_input_file = None
_auto_max_rows = None
_auto_model_choice = None

if _matches:
    _config_path = _matches[-1]
    with open(_config_path) as f:
        _cfg = json.load(f)
    _auto_run_label = _cfg.get("RUN_LABEL")
    _auto_max_rows = _cfg.get("MAX_ROWS")
    _auto_model_choice = _cfg.get("MODEL_CHOICE")
    _config_dir = _os.path.dirname(_config_path)
    _candidates = (_glob.glob(_config_dir + "/*.xlsx") +
                   _glob.glob(_config_dir + "/*.parquet"))
    if _candidates:
        _auto_input_file = _candidates[0]
    print(f"Automation override: RUN_LABEL={_auto_run_label}, MODEL={_auto_model_choice},")
    print(f"  INPUT_FILE={_auto_input_file}, MAX_ROWS={_auto_max_rows}")
    print(f"  config dir: {_config_dir}")
    print(f"  all files: {_glob.glob(_config_dir + chr(47) + chr(42))}")
else:
    print("No run_config.json found - using manual values below (standalone mode).")


In [ ]:
import glob
import os
import pandas as pd

def _read_excel_robust(path):
    """Try openpyxl first (.xlsx), fall back to xlrd for old binary .xls."""
    try:
        return pd.read_excel(path, engine="openpyxl")
    except Exception:
        return pd.read_excel(path, engine="xlrd")

RUN_LABEL = _auto_run_label or "base_qwen3b"
INPUT_FILE = _auto_input_file or "/kaggle/input/your-dataset-name/your-file.parquet"

# Map UI model key → (model family, HuggingFace model ID)
MODEL_MAP = {
    "Qwen2.5_3B":            ("qwen",  "Qwen/Qwen2.5-3B-Instruct"),
    "Indictrans2_1B":        ("it2",   "ai4bharat/indictrans2-indic-en-1B"),
    "FacebookNLLB-200_600M": ("nllb",  "facebook/nllb-200-distilled-600M"),
}
MODEL_KEY = _auto_model_choice if _auto_model_choice in MODEL_MAP else "Qwen2.5_3B"
MODEL_FAMILY, MODEL_NAME = MODEL_MAP[MODEL_KEY]
print(f"Model: {MODEL_KEY}  ({MODEL_NAME})")

MAX_ROWS_SAFEGUARD = 10  # bump manually for full runs

if not os.path.exists(INPUT_FILE):
    print(f"'{INPUT_FILE}' not found. Files currently under /kaggle/input:")
    for f in glob.glob("/kaggle/input/**/*", recursive=True):
        if os.path.isfile(f):
            print(" ", f)
    raise FileNotFoundError(f"INPUT_FILE not found: {INPUT_FILE}.")

print(f"Found input file: {INPUT_FILE}")
_check_df = (_read_excel_robust(INPUT_FILE)
             if INPUT_FILE.endswith((".xlsx", ".xls"))
             else pd.read_parquet(INPUT_FILE))
print(f"Input file has {len(_check_df):,} rows. Safeguard limit: {MAX_ROWS_SAFEGUARD:,}")
if len(_check_df) > MAX_ROWS_SAFEGUARD:
    raise ValueError(
        f"SAFETY STOP: {len(_check_df):,} rows > MAX_ROWS_SAFEGUARD={MAX_ROWS_SAFEGUARD}. "
        "Raise the limit and re-run."
    )


## 4. Load + normalize input to a common schema

Accepts `.parquet` or `.xlsx`. Required column: **`ben`** (Bengali text).
Optional columns: **`ref_en`** (reference English, if you have it),
**`source`** (provenance tag). Missing optional columns are filled with
`None`/`"unknown"` so downstream steps never break on their absence.

If the input is `.xlsx`, it is also **converted and saved as a matching
`.parquet`** for consistency with the rest of the pipeline.


In [ ]:

def load_and_normalize(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()

    if ext == ".parquet":
        df = pd.read_parquet(path)
    elif ext in (".xlsx", ".xls"):
        try:
            df = pd.read_excel(path, engine='openpyxl')
        except Exception:
            df = pd.read_excel(path, engine='xlrd')
    else:
        raise ValueError(f"Unsupported input file type: {ext}. Use .parquet or .xlsx")

    # Normalize column name casing/whitespace defensively
    df.columns = [str(c).strip().lower() for c in df.columns]

    if "ben" not in df.columns:
        # Be forgiving of common alternate names before giving up
        alt_names = {"bengali": "ben", "bn": "ben", "src_bn": "ben"}
        for alt, target in alt_names.items():
            if alt in df.columns:
                df = df.rename(columns={alt: target})
                break

    if "ben" not in df.columns:
        raise ValueError(
            f"Required column 'ben' not found. Columns present: {list(df.columns)}"
        )

    if "ref_en" not in df.columns:
        df["ref_en"] = None
    if "source" not in df.columns:
        df["source"] = "unknown"

    df = df[["ben", "ref_en", "source"]].copy()
    df["ben"] = df["ben"].astype(str).str.strip()
    df = df[df["ben"].str.len() > 0].reset_index(drop=True)

    return df, ext

input_df, input_ext = load_and_normalize(INPUT_FILE)
print(f"Loaded {len(input_df):,} rows from {INPUT_FILE}")
print(f"Schema: {list(input_df.columns)}")
input_df.head()


In [ ]:
# If the input was Excel, save a matching Parquet copy for consistency
if input_ext in (".xlsx", ".xls"):
    base_name = os.path.splitext(os.path.basename(INPUT_FILE))[0]
    CONVERTED_PARQUET_PATH = f"/kaggle/working/{base_name}__converted.parquet"
    input_df.to_parquet(CONVERTED_PARQUET_PATH, index=False)
    print(f"Converted Excel input to Parquet: {CONVERTED_PARQUET_PATH}")
else:
    print("Input was already Parquet â€” no conversion needed.")


## 5. Load Qwen2.5-3B-Instruct (base model, 4-bit)


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    AutoTokenizer, BitsAndBytesConfig,
)

print(f"Loading {MODEL_KEY} ({MODEL_NAME}) ...")

if MODEL_FAMILY == "qwen":
    # Qwen: causal LM, 4-bit on T4 or fp32 on P100/CPU.
    if torch.cuda.is_available():
        _cc = torch.cuda.get_device_capability(0)
        print(f"GPU: {torch.cuda.get_device_name(0)}  (sm_{_cc[0]}{_cc[1]})")
        _use_4bit = _cc[0] >= 7
    else:
        _use_4bit = False
        print("No CUDA GPU.")

    if _use_4bit:
        print("4-bit NF4 quantisation.")
        _bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        )
        _mkw = {"quantization_config": _bnb, "device_map": "auto"}
    else:
        print("fp32 on CPU (P100 or no GPU).")
        _mkw = {"dtype": torch.float32, "device_map": "cpu"}

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=HF_TOKEN, **_mkw)
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

elif MODEL_FAMILY == "nllb":
    # NLLB-200: direct seq2seq generation, with target language forced at generate time.
    NLLB_SRC, NLLB_TGT = "ben_Beng", "eng_Latn"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, src_lang=NLLB_SRC)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, token=HF_TOKEN, torch_dtype=_dtype)
    model = model.to(device)
    model.eval()
    NLLB_TGT_ID = tokenizer.convert_tokens_to_ids(NLLB_TGT)
    print(f"NLLB device: {device}, dtype: {_dtype}")

elif MODEL_FAMILY == "it2":
    # IndicTrans2 tokenizer expects strings prefixed with source and target language tags.
    IT2_SRC, IT2_TGT = "ben_Beng", "eng_Latn"
    if torch.cuda.is_available():
        _cc = torch.cuda.get_device_capability(0)
        _use_cuda = _cc[0] >= 7
        print(f"GPU: {torch.cuda.get_device_name(0)}  (sm_{_cc[0]}{_cc[1]})")
    else:
        _use_cuda = False
        print("No CUDA GPU.")
    device = torch.device("cuda" if _use_cuda else "cpu")
    _dtype = torch.float16 if _use_cuda else torch.float32
    if not _use_cuda:
        print("IndicTrans2 running on CPU because P100/no GPU is incompatible with this PyTorch CUDA build.")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        trust_remote_code=True,
        torch_dtype=_dtype,
    )
    model = model.to(device)
    model.eval()
    IT2_TGT_ID = tokenizer.convert_tokens_to_ids(IT2_TGT)

print("Model loaded.")

## 6. Translation prompt + batched bulk generation

Zero-shot chat-template prompting, greedy decoding (deterministic â€” useful
if you re-run this later on a fine-tuned model and want a fair comparison).


In [ ]:
@torch.no_grad()
def translate_batch(bengali_texts, max_new_tokens=256, batch_size=8):
    outputs = []

    if MODEL_FAMILY == "qwen":
        def _build_prompt(text):
            msgs = [
                {"role": "system", "content": "You are a professional Bengali to English translator. "
                 "Translate the given Bengali text into natural, fluent English. Output only the translation."},
                {"role": "user", "content": text},
            ]
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

        for i in range(0, len(bengali_texts), batch_size):
            batch = bengali_texts[i:i + batch_size]
            prompts = [_build_prompt(t) for t in batch]
            inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                               truncation=True, max_length=512).to(model.device)
            gen = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, temperature=1.0,
                                 pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
            for j, out_ids in enumerate(gen):
                new_tokens = out_ids[inputs["input_ids"][j].shape[0]:]
                outputs.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
            if (i // batch_size) % 10 == 0:
                print(f"  translated {i + len(batch)}/{len(bengali_texts)}")

    elif MODEL_FAMILY == "nllb":
        tokenizer.src_lang = NLLB_SRC
        for i in range(0, len(bengali_texts), batch_size):
            batch = bengali_texts[i:i + batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            ).to(model.device)
            gen = model.generate(
                **inputs,
                forced_bos_token_id=NLLB_TGT_ID,
                max_length=512,
                num_beams=4,
            )
            outputs.extend(t.strip() for t in tokenizer.batch_decode(gen, skip_special_tokens=True))
            if (i // batch_size) % 10 == 0:
                print(f"  translated {i + len(batch)}/{len(bengali_texts)}")

    elif MODEL_FAMILY == "it2":
        for i in range(0, len(bengali_texts), batch_size):
            batch = bengali_texts[i:i + batch_size]
            tagged_batch = [f"{IT2_SRC} {IT2_TGT} {text}" for text in batch]
            inputs = tokenizer(
                tagged_batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            ).to(model.device)
            gen = model.generate(
                **inputs,
                forced_bos_token_id=IT2_TGT_ID,
                num_beams=5,
                max_new_tokens=256,
            )
            outputs.extend(t.strip() for t in tokenizer.batch_decode(gen, skip_special_tokens=True))
            if (i // batch_size) % 10 == 0:
                print(f"  translated {i + len(batch)}/{len(bengali_texts)}")

    return outputs

print("Ready to translate.")

## 7. Run bulk translation

In [ ]:
BATCH_SIZE = 8  # lower to 4 if you hit an out-of-memory error on your GPU

bengali_inputs = input_df["ben"].tolist()
print(f"Translating {len(bengali_inputs):,} rows with {MODEL_NAME}...")

translations = translate_batch(bengali_inputs, batch_size=BATCH_SIZE)

output_df = input_df.copy()
output_df["translated_en"] = translations

print("Done.")
output_df.head()


## 8. Save output Excel file

Schema: `ben`, `ref_en`, `source`, `translated_en` â€” this is the exact
input format the **scoring notebook** expects.


In [ ]:
base_name = os.path.splitext(os.path.basename(INPUT_FILE))[0]
OUTPUT_XLSX_PATH = f"/kaggle/working/{base_name}__translated__{RUN_LABEL}.xlsx"

output_df.to_excel(OUTPUT_XLSX_PATH, index=False)
print(f"Saved translated output to: {OUTPUT_XLSX_PATH}")
print(f"Rows: {len(output_df):,}  |  Columns: {list(output_df.columns)}")
print()
print("To download to your local machine:")
print("  1. Save this notebook version (Save Version, top right).")
print("  2. Open the notebook's 'Output' tab.")
print(f"  3. Find '{os.path.basename(OUTPUT_XLSX_PATH)}' and click the download icon.")
print()
print("Feed this file into the scoring notebook as INPUT_FILE.")


## Notes

- To translate with a **fine-tuned** model later instead of the base model,
  swap Section 5's `from_pretrained` call to load your base model + apply
  your LoRA/QLoRA adapter (`PeftModel.from_pretrained(model, adapter_path)`)
  before Section 6 â€” everything else in this notebook stays the same.
- `ref_en` passes through untouched if present in the input; it's optional
  and only used later by the scoring notebook.
- Lower `BATCH_SIZE` in Section 7 if you hit a CUDA out-of-memory error.
